In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("german.data", sep=" ", header=None)

columns = [
    "Status", "Duration", "CreditHistory", "Purpose", "CreditAmount",
    "Savings", "Employment", "InstallmentRate", "Sex", "Guarantors",
    "ResidenceDuration", "Property", "Age", "OtherInstallmentPlans",
    "Housing", "ExistingCredits", "Job", "Dependents",
    "Telephone", "ForeignWorker", "Risk"
]

df.columns = columns

In [15]:
# Target: 0 = good, 1 = bad
y = df["Risk"].replace({1: 0, 2: 1})

# Features
X = df.drop(columns=["Risk"])

# Save original before encoding (IMPORTANT for grouping)
X_original = X.copy()

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, X_train_orig, X_test_orig = train_test_split(
    X, y, X_original, test_size=0.2, random_state=42
)

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score, recall_score

print("Overall Accuracy:", accuracy_score(y_test, y_pred))
print("Overall Recall:", recall_score(y_test, y_pred))

Overall Accuracy: 0.795
Overall Recall: 0.5932203389830508


In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

# Cost:
# FP (bad → predicted good) = 1
# FN (good → predicted bad) = 5

cost = (FP * 1) + (FN * 5)

print("Confusion Matrix:\n", cm)
print("Total Cost:", cost)

Confusion Matrix:
 [[124  17]
 [ 24  35]]
Total Cost: 137


In [21]:
# Add predictions to original test set
X_test_orig = X_test_orig.copy()
X_test_orig["y_true"] = y_test.values
X_test_orig["y_pred"] = y_pred

# Age groups
X_test_orig["AgeGroup"] = X_test_orig["Age"].apply(
    lambda x: "<30" if x < 30 else ">=30"
)

In [22]:
def evaluate_group(data, column):
    print(f"\n--- {column} ---")

    for group in data[column].unique():
        subset = data[data[column] == group]

        acc = accuracy_score(subset["y_true"], subset["y_pred"])
        rec = recall_score(subset["y_true"], subset["y_pred"])

        print(f"{group}: Accuracy={acc:.3f}, Recall={rec:.3f}")

In [23]:
#Bias analysis by Age group
#Explicit features
evaluate_group(X_test_orig, "Sex")
evaluate_group(X_test_orig, "AgeGroup")


--- Sex ---
A92: Accuracy=0.821, Recall=0.588
A93: Accuracy=0.806, Recall=0.613
A94: Accuracy=0.700, Recall=0.250
A91: Accuracy=0.750, Recall=0.714

--- AgeGroup ---
<30: Accuracy=0.797, Recall=0.737
>=30: Accuracy=0.794, Recall=0.525


In [24]:
#Proxy features
evaluate_group(X_test_orig, "Job")
evaluate_group(X_test_orig, "Housing")
evaluate_group(X_test_orig, "CreditHistory")


--- Job ---
A173: Accuracy=0.824, Recall=0.636
A172: Accuracy=0.740, Recall=0.556
A174: Accuracy=0.800, Recall=0.667
A171: Accuracy=0.600, Recall=0.000

--- Housing ---
A152: Accuracy=0.779, Recall=0.425
A151: Accuracy=0.846, Recall=1.000
A153: Accuracy=0.810, Recall=0.833

--- CreditHistory ---
A32: Accuracy=0.782, Recall=0.588
A31: Accuracy=0.727, Recall=0.833
A34: Accuracy=0.898, Recall=0.545
A33: Accuracy=0.533, Recall=0.333
A30: Accuracy=0.800, Recall=1.000


The model achieved an overall accuracy of 0.795 and recall of 0.593, indicating moderate performance but some difficulty in correctly identifying high-risk (bad credit) applicants. When analyzing performance across groups, clear disparities are observed. For explicit sensitive features, differences across sex groups are noticeable, particularly for group A94, which has significantly lower recall (0.250), meaning the model frequently fails to identify risky individuals in this group. Age-based analysis also reveals bias: while accuracy is similar, recall is much higher for applicants under 30 (0.737) compared to those aged 30 and above (0.525), suggesting that older individuals are more likely to be misclassified as low-risk.
More pronounced disparities appear in proxy features. For example, in the Job category, group A171 (unemployed/unskilled non-resident) has a recall of 0.000, indicating that the model completely fails to detect risky applicants in this group. Similarly, variations in Housing and Credit History show inconsistent performance, with some groups achieving perfect recall (1.000) while others perform poorly (e.g., A33 with recall 0.333). These results demonstrate that proxy features reveal additional and even stronger bias compared to explicit features like sex and age, likely because they capture underlying socio-economic conditions.
These differences may arise from imbalanced data distribution and inherent correlations between financial status, employment, and credit behavior. This is critical in real-world credit scoring systems, as such biases can lead to unfair loan approvals or rejections, reinforcing financial inequality and potentially causing ethical and legal issues.